# Tutorial: Environment Setup and Repo Orientation

Detailed step-by-step workflow notebook for this SD-dMFA repository.


## Audience, Prerequisites, Outcomes

**Audience**
- Developers onboarding to SD-dMFA workflows.

**Prerequisites**
- Python 3.11+ environment for this repo.
- `pip install -e ".[dev]"` completed.
- Notebook executed from repository root or a subfolder.

**Outcomes**
- Verify local Python and dependency state.
- Inspect repository structure and main script entry points.
- Understand outputs/data/docs locations before running model code.


## Outline

1. Verify Python/runtime context
2. Check package imports and versions
3. Map repository structure
4. Inspect docs and scripts index
5. Confirm config files available for use


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
from pathlib import Path
from textwrap import dedent

import pandas as pd
import matplotlib.pyplot as plt

try:
    from crm_model.common.io import load_run_config
except Exception:
    load_run_config = None

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)


In [ ]:
DRY_RUN = False
RUN_HEAVY = False
RUN_PLOTS = False
RUN_CALIBRATION = False
RUN_AUDIT = False

CONFIG = "configs/runs/mvp.yml"
EXAMPLE_VARIANT = "baseline"


In [ ]:
def find_repo_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / "configs").exists() and (cand / "src").exists():
            return cand
    raise RuntimeError("Could not locate repo root from current working directory.")


def sh(cmd: str, *, cwd: Path, check: bool = True) -> subprocess.CompletedProcess | None:
    print(f"$ {cmd}")
    if DRY_RUN:
        return None
    cp = subprocess.run(cmd, cwd=str(cwd), shell=True, text=True, capture_output=True)
    if cp.stdout.strip():
        print(cp.stdout)
    if cp.stderr.strip():
        print(cp.stderr)
    if check and cp.returncode != 0:
        raise RuntimeError(f"Command failed ({cp.returncode}): {cmd}")
    return cp


def latest_dir(base: Path) -> Path | None:
    if not base.exists():
        return None
    cands = [p for p in base.iterdir() if p.is_dir() and p.name != "_archive"]
    return sorted(cands)[-1] if cands else None


def load_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        print(f"Missing: {path}")
        return pd.DataFrame()
    return pd.read_csv(path)


REPO = find_repo_root()
CONFIG_PATH = (REPO / CONFIG).resolve()
CONFIG_STEM = CONFIG_PATH.stem
print("Repo:", REPO)
print("Config:", CONFIG_PATH)


## Step 1: Verify runtime basics


In [ ]:
import sys
print("Python executable:", sys.executable)
print("Python version:", sys.version)


## Step 2: Confirm core package imports


In [ ]:
import numpy as np
import pandas as pd
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
if load_run_config is None:
    print('crm_model import failed: run pip install -e \".[dev]\"')
else:
    print("crm_model import: OK")


## Step 3: Inspect top-level repo layout


In [ ]:
for name in ["configs", "data", "src", "scripts", "outputs", "docs", "tests"]:
    p = REPO / name
    print(f"{name:10s}", "exists" if p.exists() else "missing")


## Step 4: Inspect run configs and scenario packs


In [ ]:
runs = sorted((REPO / "configs" / "runs").glob("*.yml"))
print("Run configs:")
for p in runs:
    print(" -", p.relative_to(REPO))

for folder in ["mvp", "r_strategies"]:
    sdir = REPO / "configs" / "scenarios" / folder
    files = sorted(sdir.glob("*.yml"))
    print(f"\nScenario folder {folder}: {len(files)} files")
    for p in files[:8]:
        print(" -", p.name)


## Step 5: Verify executable scripts


In [ ]:
scripts = [
    "scripts/run_one.py",
    "scripts/run_batch.py",
    "scripts/validation/lint_run_configs.py",
    "scripts/validation/validate_exogenous_inputs.py",
    "scripts/analysis/compare_scenarios.py",
    "scripts/analysis/plots/plot_scenario_subset_panels.py",
    "scripts/calibration/calibrate_model.py",
]
for s in scripts:
    print(s, "OK" if (REPO / s).exists() else "MISSING")


## Pitfalls

- Running from outside repo root can break relative paths.
- In VS Code zsh, strict `set -u` can fail due to prompt vars; keep that in script files, not interactive shell.


## Exercises

1. Repeat this workflow with `CONFIG=configs/runs/r-strategies.yml`.
2. Record one thing that changed and why.
3. Add one guardrail/check specific to your team workflow.


In [ ]:
# Exercise answer scaffold
pass
